# Прогноз количества запросов на следующий час

Ноутбук оставляет только основной пайплайн: подготовка почасового ряда, признаки, модели `baseline`, `linear`, `rf`, `cat`, `arima`, таблицу метрик и два финальных графика.


In [169]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from catboost import CatBoostRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (15, 6)
plt.rcParams["axes.grid"] = True


## Настройки


In [170]:
DATA_PATH = "df_with_cat.csv"
city_ids = [34, 16, 35, 6, 7, 42, 1, 38, 21, 66, 41, 27]
CITY_ID = city_ids[10]

TEST_SIZE = 0.20
VALID_SIZE = 0.20
RANDOM_STATE = 42

RF_TREES = 500
CAT_ITERATIONS = 3000


## Подготовка данных


In [171]:
events = pd.read_csv(DATA_PATH)

In [172]:
events["model_response_timestamp"] = pd.to_datetime(
    events["model_response_timestamp"],
    unit="s",
)
events["date_hour"] = events["model_response_timestamp"].dt.floor("h")

hourly = (
    events.groupby(["number", "date_hour"])
    .size()
    .reset_index(name="count")
)

city_hourly = hourly.loc[hourly["number"] == CITY_ID, ["date_hour", "count"]]
all_hours = pd.date_range(
    city_hourly["date_hour"].min(),
    city_hourly["date_hour"].max(),
    freq="h",
)

base_df = (
    city_hourly.set_index("date_hour")
    .reindex(all_hours, fill_value=0)
    .rename_axis("date_hour")
    .reset_index()
)

base_df = base_df.sort_values("date_hour").reset_index(drop=True)

base_df.head()


,date_hour,count
0,2026-02-13 21:00:00,3
1,2026-02-13 22:00:00,3
2,2026-02-13 23:00:00,5
3,2026-02-14 00:00:00,0
4,2026-02-14 01:00:00,0


In [173]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.sort_values("date_hour").reset_index(drop=True).copy()

    result["hour"] = result["date_hour"].dt.hour
    result["day"] = result["date_hour"].dt.day
    result["month"] = result["date_hour"].dt.month
    result["day_of_week"] = result["date_hour"].dt.dayofweek
    result["is_weekend"] = result["day_of_week"].isin([5, 6]).astype(int)

    result["hour_sin"] = np.sin(2 * np.pi * result["hour"] / 24)
    result["hour_cos"] = np.cos(2 * np.pi * result["hour"] / 24)

    for day in range(7):
        result[f"day_of_week_{day}"] = (result["day_of_week"] == day).astype(int)

    return result


def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.sort_values("date_hour").reset_index(drop=True).copy()

    result["lag_1"] = result["count"].shift(1)
    result["lag_2"] = result["count"].shift(2)
    result["lag_24"] = result["count"].shift(24)

    result["rolling_mean_3"] = result["count"].shift(1).rolling(3).mean()
    result["rolling_mean_6"] = result["count"].shift(1).rolling(6).mean()
    result["rolling_mean_24"] = result["count"].shift(1).rolling(24).mean()

    return result


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    return add_lag_features(add_time_features(df))


In [174]:
featured_df = build_features(base_df)
featured_df["predict_1h"] = featured_df["count"].shift(-1)
featured_df["target_hour"] = featured_df["date_hour"] + pd.Timedelta(hours=1)

excluded_dates = featured_df["month"].eq(2) & featured_df["day"].isin([14, 15, 25])
recursive_base_df = base_df.loc[~excluded_dates.to_numpy()].reset_index(drop=True)
featured_df = featured_df.loc[~excluded_dates].reset_index(drop=True)

day_columns = [f"day_of_week_{day}" for day in range(7)]
feature_cols = [
    "count",
    "hour_sin",
    "hour_cos",
    "lag_1",
    "lag_2",
    "lag_24",
    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_mean_24",
    "is_weekend",
    *day_columns,
]

model_df = featured_df.dropna(subset=feature_cols + ["predict_1h"]).reset_index(drop=True)
model_df[["date_hour", "target_hour", "count", "predict_1h"]].head()


,date_hour,target_hour,count,predict_1h
0,2026-02-16 00:00:00,2026-02-16 01:00:00,0,0.0
1,2026-02-16 01:00:00,2026-02-16 02:00:00,0,0.0
2,2026-02-16 02:00:00,2026-02-16 03:00:00,0,0.0
3,2026-02-16 03:00:00,2026-02-16 04:00:00,0,0.0
4,2026-02-16 04:00:00,2026-02-16 05:00:00,0,0.0


## Разделение на train / validation / test


In [175]:
split_index = int(len(model_df) * (1 - TEST_SIZE))
train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()

valid_index = int(len(train_df) * (1 - VALID_SIZE))
cat_train_df = train_df.iloc[:valid_index].copy()
cat_valid_df = train_df.iloc[valid_index:].copy()

X_train = train_df[feature_cols]
y_train = train_df["predict_1h"]

X_cat_train = cat_train_df[feature_cols]
y_cat_train = cat_train_df["predict_1h"]
X_cat_valid = cat_valid_df[feature_cols]
y_cat_valid = cat_valid_df["predict_1h"]

X_test = test_df[feature_cols]
y_test = test_df["predict_1h"]

print(f"train: {X_train.shape}")
print(f"cat validation: {X_cat_valid.shape}")
print(f"test: {X_test.shape}")


train: (553, 17)
cat validation: (111, 17)
test: (139, 17)


## Метрики


In [176]:
scores = []
preds_df = pd.DataFrame({
    "date_hour": test_df["target_hour"].values,
    "true": y_test.values,
})


def save_prediction(name: str, y_true, y_pred) -> None:
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    preds_df[name] = y_pred

    scores.append({
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
    })


## Baseline

Наивный прогноз: считаем, что в следующий час будет столько же запросов, сколько в текущий.


In [177]:
y_pred_baseline = X_test["count"].values
save_prediction("baseline", y_test, y_pred_baseline)


## Linear Regression


In [178]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred_linear = linear_model.predict(X_test)
save_prediction("linear", y_test, y_pred_linear)


## Random Forest


In [179]:
rf_model = RandomForestRegressor(
    n_estimators=RF_TREES,
    max_depth=20,
    min_samples_split=20,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
joblib.dump(rf_model, f"rf_total_model_{CITY_ID}.joblib")

y_pred_rf = rf_model.predict(X_test)
save_prediction("rf", y_test, y_pred_rf)


## CatBoost


In [180]:
cat_model = CatBoostRegressor(
    iterations=CAT_ITERATIONS,
    learning_rate=0.01,
    depth=8,
    l2_leaf_reg=3,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=RANDOM_STATE,
    early_stopping_rounds=100,
    verbose=200,
)
cat_model.fit(
    X_cat_train,
    y_cat_train,
    eval_set=(X_cat_valid, y_cat_valid),
    use_best_model=True,
)

y_pred_cat = cat_model.predict(X_test)
save_prediction("cat", y_test, y_pred_cat)


0:	learn: 1.9127475	test: 1.8406482	best: 1.8406482 (0)	total: 3.05ms	remaining: 9.15s
200:	learn: 1.4826912	test: 1.7251732	best: 1.7222281 (171)	total: 291ms	remaining: 4.06s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.722228098
bestIteration = 171

Shrink model to first 172 iterations.


## Сводная таблица


In [181]:
scores_df = pd.DataFrame(scores).sort_values("RMSE").reset_index(drop=True)
display(scores_df)


,model,MAE,RMSE,R2,MAPE
0,cat,1.324000,1.597486,0.039497,2.748821e+15
1,rf,1.336792,1.608722,0.025938,2.740862e+15
2,linear,1.367399,1.664390,-0.042641,2.687293e+15
3,baseline,1.453237,2.155802,-0.749211,2.170800e+15
